## Downscaling with the ViT model

This notebook showcases a simple application of deep4downscaling for the statistical downscaling of precipitation using the Vision Transformer (ViT) architecture [1]. In particular, we use the NoisyViT variant [2], which injects noise into the transformer encoder to generate stochastic outputs, enabling training with the Continuous Ranked Probability Score (CRPS) loss function. To do so, we will implement the following actions:

- Define and train the NoisyViT architecture.
- Downscale and evaluate results over a test period.

### Train the model

In [1]:
DATA_PATH = './data/input'
FIGURES_PATH = './figures'
MODELS_PATH = './models'

In [2]:
import xarray as xr
import torch
from torch.utils.data import DataLoader, random_split
import numpy as np

import deep4downscaling.viz
import deep4downscaling.trans
import deep4downscaling.deep.loss
import deep4downscaling.deep.utils
import deep4downscaling.deep.models
import deep4downscaling.deep.train
import deep4downscaling.deep.pred

We will begin by loading the predictor. In this case, we select various large-scale variables from ERA5 at different height levels. These variables are already stored in a NetCDF file, the standard data format for deep4downscaling. Unfortunately, due to GitHub's size restrictions, we are unable to upload these files to the repository. However, the following cells provide an overview of the data, making it straightforward to reproduce this notebook with a similar file.

In [8]:
predictor_filename = f'{DATA_PATH}/ERA5_NorthAtlanticRegion_1-5dg_full.nc'
predictor = xr.open_dataset(predictor_filename)

In [9]:
predictor.load()

<xarray.Dataset> Size: 1GB
Dimensions:  (lon: 42, lat: 31, time: 16071)
Coordinates:
  * lon      (lon) float64 336B -39.0 -37.5 -36.0 -34.5 ... 18.0 19.5 21.0 22.5
  * lat      (lat) float64 248B 23.5 25.0 26.5 28.0 29.5 ... 64.0 65.5 67.0 68.5
  * time     (time) datetime64[ns] 129kB 1979-01-01 1979-01-02 ... 2022-12-31
Data variables: (12/13)
    t500     (time, lat, lon) float32 84MB 262.4 262.9 263.0 ... 245.3 244.8
    t700     (time, lat, lon) float32 84MB 279.7 280.2 280.8 ... 260.3 260.6
    t850     (time, lat, lon) float32 84MB 285.2 284.9 285.2 ... 266.3 266.8
    q500     (time, lat, lon) float32 84MB 0.0003375 0.0002513 ... 0.0005094
    q700     (time, lat, lon) float32 84MB 0.001221 0.0008963 ... 0.001434
    q850     (time, lat, lon) float32 84MB 0.005046 0.005577 ... 0.002642
    ...       ...
    v700     (time, lat, lon) float32 84MB 6.084 4.734 5.369 ... 18.41 17.4
    v850     (time, lat, lon) float32 84MB 3.22 3.965 1.448 ... 14.55 16.53
    u500     (time, lat, lon) float32 84MB 10.56 7.765 5.109 ... 2.195 1.534
    u700     (time, lat, lon) float32 84MB 9.172 7.134 3.817 ... 0.03401 0.2164
    u850     (time, lat, lon) float32 84MB 0.2598 0.2815 ... -4.335 -2.634
    msl      (time, lat, lon) float32 84MB 1.016e+05 1.016e+05 ... 9.853e+04
Attributes:
    Conventions:  CF-1.6
    history:      2023-06-06 07:20:26 GMT by grib_to_netcdf-2.25.1: /opt/ecmw...

#### Adjusting predictor dimensions for the ViT

The ViT model splits the input into fixed-size, non-overlapping patches for building token embeddings. This imposes constraints on the spatial dimensions of the predictor:

- Both the height and width of the input must be **divisible by the patch size**.
- The spatial resolution of the output (predictand) must be a **multiple of the number of input tokens** per spatial dimension (i.e., `height / patch_size`).

In practice, this means the predictor grid may need to be extended to meet these constraints. Here, we extend the latitude dimension to 32 grid points by reindexing with the nearest available value. We also need to subset the longitude to span 32 grid points.

In [10]:
predictor = predictor.sel(lon=slice(-24, 22.5))

In [11]:
lat = predictor.lat.values
dlat = np.diff(lat).mean()
n_extra = 32 - lat.size
new_lat = np.concatenate([lat, lat[-1] + dlat * np.arange(1, n_extra + 1)])
predictor = predictor.reindex(lat=new_lat, method='nearest')

In [12]:
predictor

<xarray.Dataset> Size: 856MB
Dimensions:  (lon: 32, lat: 32, time: 16071)
Coordinates:
  * lon      (lon) float64 256B -24.0 -22.5 -21.0 -19.5 ... 18.0 19.5 21.0 22.5
  * lat      (lat) float64 256B 23.5 25.0 26.5 28.0 29.5 ... 65.5 67.0 68.5 70.0
  * time     (time) datetime64[ns] 129kB 1979-01-01 1979-01-02 ... 2022-12-31
Data variables: (12/13)
    t500     (time, lat, lon) float32 66MB 264.3 264.3 264.2 ... 245.3 244.8
    t700     (time, lat, lon) float32 66MB 283.3 282.9 282.7 ... 260.3 260.6
    t850     (time, lat, lon) float32 66MB 289.5 289.9 289.9 ... 266.3 266.8
    q500     (time, lat, lon) float32 66MB 0.0001745 0.0001698 ... 0.0005094
    q700     (time, lat, lon) float32 66MB 0.0003933 0.0003953 ... 0.001434
    q850     (time, lat, lon) float32 66MB 0.001516 0.00127 ... 0.002642
    ...       ...
    v700     (time, lat, lon) float32 66MB 3.518 2.116 0.6025 ... 18.41 17.4
    v850     (time, lat, lon) float32 66MB 2.824 2.14 1.582 ... 14.55 16.53
    u500     (time, lat, lon) float32 66MB -7.773 -8.511 -9.043 ... 2.195 1.534
    u700     (time, lat, lon) float32 66MB -7.559 -7.929 ... 0.03401 0.2164
    u850     (time, lat, lon) float32 66MB -7.11 -7.462 -7.876 ... -4.335 -2.634
    msl      (time, lat, lon) float32 66MB 1.018e+05 1.019e+05 ... 9.853e+04
Attributes:
    Conventions:  CF-1.6
    history:      2023-06-06 07:20:26 GMT by grib_to_netcdf-2.25.1: /opt/ecmw...

The predictand is an `xarray.Dataset` containing a single variable (the target). In this notebook, we will focus on downscaling accumulated precipitation over the region of peninsular Spain.

In [13]:
predictand_filename = f'{DATA_PATH}/pr_AEMET.nc'
predictand = xr.open_dataset(predictand_filename)

In [14]:
predictand

<xarray.Dataset> Size: 10GB
Dimensions:  (lon: 400, lat: 251, time: 25933)
Coordinates:
  * lon      (lon) float64 3kB -13.18 -13.12 -13.07 -13.02 ... 6.675 6.725 6.775
  * lat      (lat) float64 2kB 33.48 33.52 33.57 33.62 ... 45.87 45.92 45.97
  * time     (time) datetime64[ns] 207kB 1951-01-01 1951-01-02 ... 2021-12-31
Data variables:
    pr       (time, lat, lon) float32 10GB ...

#### Adjusting predictand dimensions for the ViT

Similarly to the predictor, the predictand grid must also be adjusted. The ViT model internally computes the output spatial dimensions as `H_out = W_out = sqrt(num_outputs)`, meaning the **total number of output grid points must be a perfect square** (i.e., the output grid must be square). Additionally, `H_out` must be divisible by the number of input tokens per dimension.

Here, we extend the latitude to 256 grid points and subset the longitude to 256.

In [15]:
predictand = predictand.sel(lon=slice(-9.425, 3.375))

In [16]:
predictand.load()

lat = predictand.lat.values
dlat = np.diff(lat).mean()

n_extra = 256 - lat.size
new_lat = np.concatenate([lat, lat[-1] + dlat * np.arange(1, n_extra + 1)])

predictand = predictand.reindex(lat=new_lat)

In [17]:
predictand

<xarray.Dataset> Size: 7GB
Dimensions:  (lon: 256, lat: 256, time: 25933)
Coordinates:
  * lon      (lon) float64 2kB -9.425 -9.375 -9.325 -9.275 ... 3.225 3.275 3.325
  * lat      (lat) float64 2kB 33.48 33.52 33.57 33.62 ... 46.12 46.17 46.22
  * time     (time) datetime64[ns] 207kB 1951-01-01 1951-01-02 ... 2021-12-31
Data variables:
    pr       (time, lat, lon) float32 7GB nan nan nan nan ... nan nan nan nan

Deep4downscaling includes several common preprocessing techniques used in statistical downscaling, such as removing NaN values, aligning datasets (e.g., across time), bias adjustment, and standardization.

In [18]:
predictor = deep4downscaling.trans.remove_days_with_nans(predictor)

predictor, predictand = deep4downscaling.trans.align_datasets(predictor, predictand, 'time')

There are no observations containing null values


To adhere to the standard training/validation scheme in the machine learning field, we divide the predictors and predictand into training and test sets.

In [19]:
years_train = ('1980', '1983') # ('1980', '2010')
years_test = ('2011', '2020')

x_train = predictor.sel(time=slice(*years_train))
y_train = predictand.sel(time=slice(*years_train))

x_test = predictor.sel(time=slice(*years_test))
y_test = predictand.sel(time=slice(*years_test))

Before feeding the predictors to the deep learning model, we standardize them to have a mean of zero and a standard deviation of one. This is done using the `deep4downscaling.trans.standardize` function, which stores the training statistics to later standardize the test set consistently.

In [22]:
from importlib import reload
reload(deep4downscaling.trans)
x_train_stand = deep4downscaling.trans.standardize(data_ref=x_train, data=x_train)

The mask identifies valid (non-NaN) grid points and is required during the prediction phase to convert the raw model output back into a properly formatted `xr.Dataset` with the correct spatial structure.

In [24]:
y_mask = deep4downscaling.trans.compute_valid_mask(y_train)

All deep learning models implemented in deep4downscaling flatten their output into a vector, standardizing its dimensions to the shape `(time, grid point)`.

Note that, unlike CNN-based models such as DeepESD, the ViT requires the full spatial grid to be preserved (including NaN grid points). This is because the model internally reconstructs a square spatial output from patch-level predictions, so filtering out grid points would break the spatial structure.

In [25]:
y_train_stack = y_train.stack(gridpoint=('lat', 'lon'))

The deep4downscaling library includes various loss functions for training deep learning models. In this notebook, we use the CRPS (Continuous Ranked Probability Score) loss function [2]. The CRPS loss requires the model to produce multiple stochastic outputs (ensemble members), which is why we use the NoisyViT architecture. The `ignore_nans` flag ensures that NaN grid points in the target are properly handled during training.

In [26]:
loss_function = deep4downscaling.deep.loss.CRPSLoss(ignore_nans=True)

NetCDF is not well-suited for use with PyTorch (or for converting to the `torch.Tensor` type). In contrast, NumPy is.

In [27]:
x_train_stand_arr = deep4downscaling.trans.xarray_to_numpy(x_train_stand)
y_train_arr = deep4downscaling.trans.xarray_to_numpy(y_train_stack)

With our data now in the numpy format, we can create the `torch.Dataset` and `torch.DataLoader` to feed batches of data to the deep learning model during training.

In [28]:
train_dataset = deep4downscaling.deep.utils.StandardDataset(x=x_train_stand_arr,
                                                            y=y_train_arr)

train_dataset, valid_dataset = random_split(train_dataset,
                                            [0.9, 0.1])

batch_size = 2

train_dataloader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size,
                              shuffle=True)

#### NoisyViT model parameters

In this notebook, we will train the NoisyViT architecture for statistical downscaling. The NoisyViT extends the base ViT [1] by injecting noise into the transformer encoder, enabling stochastic outputs as proposed in [2]. During training, the model generates multiple ensemble members per input, which are used to compute the CRPS loss. During inference, each forward pass produces a single stochastic realization.

The NoisyViT shares all the base ViT parameters and adds noise-specific ones:

| Parameter | Type | Description |
|-----------|------|-------------|
| `x_shape` | tuple | Shape of the input data `(batch, channels, height, width)`. Both height and width must be divisible by `patch_size`. |
| `y_shape` | tuple | Shape of the output data `(batch, num_outputs)`. `num_outputs` must be a perfect square (since the model assumes a square output grid). |
| `patch_size` | int | Size of the non-overlapping patches extracted from the input. Must divide both the height and width of the input. |
| `dim` | int | Dimension of the token embeddings. Must be divisible by `num_heads`. |
| `depth` | int | Number of transformer encoder blocks. |
| `num_heads` | int | Number of attention heads in each transformer block. |
| `mlp_dim` | int | Hidden dimension of the MLP within each transformer block. |
| `noise_channels` | int | Number of noise channels injected into the encoder. Controls the amount of stochasticity. |
| `noise_dim` | int | Dimension of the noise embeddings passed to the conditional layer normalization in each transformer block. |
| `members_for_training` | int | Number of ensemble members generated per input during training (default: 2). More members improve the CRPS estimate but increase memory usage. |
| `dropout` | float | Dropout probability (default: 0.0). |
| `orog` | torch.Tensor | Optional orography tensor of shape `(H_out, W_out)`. If provided, orography patches are embedded and added to the token representations before decoding. |
| `overlap` | int | Overlap between decoded patches (default: 0). When > 0, uses a Hann window and overlap-add reconstruction to reduce artifacts at patch boundaries, which is especially useful when noise is injected independently per patch. |
| `last_relu` | bool | If True, applies a ReLU activation to the final output (default: False). Useful for variables that must be non-negative (e.g., precipitation). |

#### Data format requirements and limitations

The ViT imposes specific constraints on the input and output data:

- **Input**: must be 4D with shape `(batch, channels, height, width)`. Both `height` and `width` must be divisible by `patch_size`.
- **Output**: must be 2D with shape `(batch, num_outputs)`. The model computes `H_out = W_out = sqrt(num_outputs)`, so `num_outputs` must be a **perfect square** (i.e., the output grid must be square).
- The output spatial resolution (`H_out`) must be divisible by the number of tokens per dimension (`height / patch_size`), since the model upscales each token to a local patch of the output grid.
- Unlike CNN-based models (e.g., DeepESD), **NaN grid points cannot be filtered out** before training, since the ViT reconstructs a full square spatial grid from its token-level predictions. Instead, the loss function handles NaN values via the `ignore_nans` flag.

In [29]:
model_name = 'NoisyViT_pr'
model = deep4downscaling.deep.models.NoisyViT(x_shape=x_train_stand_arr.shape,
                                              y_shape=y_train_arr.shape,
                                              patch_size=4,
                                              dim=768,
                                              depth=2,
                                              num_heads=2,
                                              mlp_dim=3072,
                                              noise_channels=2,
                                              noise_dim=2,
                                              members_for_training=2,
                                              orog=None,
                                              last_relu=True)

We set the typical training hyperparameters, as is commonly done in PyTorch.

In [30]:
num_epochs = 10000
patience_early_stopping = 20

learning_rate = 0.00001
optimizer = torch.optim.Adam(model.parameters(),
                             lr=learning_rate)

Deep learning models can run on either CPU or GPU devices. We provide the corresponding `.yml` environment files (`deep4downscaling/requirement`) to set up a basic Conda environment for running deep4downscaling on both CPU and GPU.

In [31]:
device = ('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)

NoisyViT(
  (patch_embedding): Conv2d(13, 768, kernel_size=(4, 4), stride=(4, 4))
  (dropout_emb): Dropout(p=0.0, inplace=False)
  (noise_embedding): NoiseEmbedding(
    (mlp): Sequential(
      (0): Linear(in_features=2, out_features=2, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=2, out_features=2, bias=True)
    )
    (norm): LayerNorm((2,), eps=1e-05, elementwise_affine=True)
  )
  (transformer_blocks): ModuleList(
    (0-1): 2 x TransformerBlockCLN(
      (norm1): ConditionalLayerNorm(
        (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=False)
        (gamma): Linear(in_features=2, out_features=768, bias=True)
        (beta): Linear(in_features=2, out_features=768, bias=True)
      )
      (attn): MultiHeadAttention(
        (qkv_proj): Linear(in_features=768, out_features=2304, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (mlp): Sequent

Deep4downscaling provides the `deep4downscaling.deep.train.standard_training_loop`, which implements a basic training routine. Models are saved based on their performance on a validation set through an early stopping mechanism.

In [33]:
from importlib import reload
reload(deep4downscaling.deep.train)

train_loss, val_loss = deep4downscaling.deep.train.standard_training_loop(model=model, model_name=model_name, model_path=MODELS_PATH,
                                                                          device=device, num_epochs=num_epochs,
                                                                          loss_function=loss_function, optimizer=optimizer,
                                                                          train_data=train_dataloader, valid_data=valid_dataloader,
                                                                          patience_early_stopping=patience_early_stopping,
                                                                          clip_gradients_norm=1.0)

### Downscale the test set

Once a model has been trained and saved as a `.pt` file, it is easy to compute predictions on a new set of predictors. In this example, we will compute predictions on the test set, which was subset a few cells above.

Since the NoisyViT is a stochastic model, we can generate an ensemble of predictions by setting the `ensemble_size` parameter. Each forward pass produces a different realization due to the injected noise.

In [36]:
model.load_state_dict(torch.load(f'{MODELS_PATH}/{model_name}.pt'))

x_test_stand = deep4downscaling.trans.standardize(data_ref=x_train, data=x_test)

pred_test = deep4downscaling.deep.pred.compute_preds_standard(
                                x_data=x_test_stand, model=model,
                                device=device, var_target='pr',
                                mask=y_mask, ensemble_size=5,
                                batch_size=16)

In [39]:
pred_test

<xarray.Dataset> Size: 5GB
Dimensions:  (member: 5, time: 3653, lat: 256, lon: 256)
Coordinates:
  * lat      (lat) float64 2kB 33.48 33.52 33.57 33.62 ... 46.12 46.17 46.22
  * lon      (lon) float64 2kB -9.425 -9.375 -9.325 -9.275 ... 3.225 3.275 3.325
  * time     (time) datetime64[ns] 29kB 2011-01-01 2011-01-02 ... 2020-12-31
Dimensions without coordinates: member
Data variables:
    pr       (member, time, lat, lon) float32 5GB nan nan nan ... nan nan nan

### References

[1] Dosovitskiy, A., Beyer, L., Kolesnikov, A., Weissenborn, D., Zhai, X., Unterthiner, T., ... & Houlsby, N. (2020). An image is worth 16x16 words: Transformers for image recognition at scale. arXiv preprint arXiv:2010.11929.

[2] Lang, S., Alexe, M., Clare, M. C., Roberts, C., Adewoyin, R., Bouallègue, Z. B., ... & Leutbecher, M. (2024). AIFS-CRPS: ensemble forecasting using a model trained with a loss function based on the continuous ranked probability score. arXiv preprint arXiv:2412.15832.